# Day 2 · 이론 ② — 확률적 경사 하강법 · 추가 실습

**공식 실습:** `4-2.ipynb`  
**추가 실습:** `4-2_extra_practice.ipynb`

교안의 **실습 A(5-4) → B(6-5) → C(7-4) → 8장 검증 실험 → 9장 손실 함수 비교** 순서입니다. 위에서부터 실행합니다. 해석·핵심 질문·정답은 강의교안을 참고합니다.

## 0. 공통 데이터 준비 

In [ ]:
import pandas as pd

fish = pd.read_csv('https://bit.ly/fish_csv_data')

fish_input = fish[['Weight', 'Length', 'Diagonal', 'Height', 'Width']]
fish_target = fish['Species']

In [ ]:
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = train_test_split(
    fish_input, fish_target, random_state=42
)

In [ ]:
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
ss.fit(train_input)

train_scaled = ss.transform(train_input)
test_scaled = ss.transform(test_input)

## 실습 A — 정확도가 같아도 손실이 다른 이유 (교안 5-4)

### A-1. 정답 클래스 확률에 따른 로지스틱 손실

In [ ]:
import numpy as np
import pandas as pd

correct_probabilities = np.array([0.9, 0.6, 0.1])

# 로지스틱 손실 = -log(정답 클래스의 확률)
losses = -np.log(correct_probabilities)

results_a = pd.DataFrame({
    '정답 클래스 확률': correct_probabilities,
    '로지스틱 손실': np.round(losses, 3)
})
print(results_a.to_string(index=False))

### A-2. 같은 클래스를 맞혀도 손실이 다른 경우

In [ ]:
positive_probabilities = np.array([0.6, 0.9])

# 정답이 1이므로 양성 클래스 확률이 곧 정답 클래스 확률
predictions = (positive_probabilities > 0.5).astype(int)
losses = -np.log(positive_probabilities)

results_a_compare = pd.DataFrame({
    '양성 클래스 확률': positive_probabilities,
    '예측 클래스': predictions,
    '손실': np.round(losses, 3)
})
print(results_a_compare.to_string(index=False))

### A-3. 응용

A-1 셀의 `correct_probabilities`에 `0.99`, `0.01`을 **추가하여 다시 실행**합니다.

## 실습 B — 경사 하강법으로 계수와 절편 갱신 (교안 6-5)

### B-1. 초기 예측·손실

In [ ]:
x = 1.0
y = 1.0

w = 0.0
b = 0.0
learning_rate = 0.1  # 이 값만 변경해 다시 실행해 보세요.

# 점수 → 시그모이드 확률 → 로지스틱 손실
z = w * x + b
p = 1 / (1 + np.exp(-z))
loss = -np.log(p)

print(f'점수(z): {z:.4f}')
print(f'양성 클래스 확률: {p:.4f}')
print(f'로지스틱 손실: {loss:.4f}')

### B-2. 경사 계산

In [ ]:
# 규제항을 제외한 로지스틱 손실의 경사
gradient_w = (p - y) * x
gradient_b = p - y

print(f'계수 경사: {gradient_w:.4f}')
print(f'절편 경사: {gradient_b:.4f}')

### B-3. 계수·절편 갱신

In [ ]:
# 새 값 = 현재 값 - 학습률 × 경사
w_new = w - learning_rate * gradient_w
b_new = b - learning_rate * gradient_b

print(f'갱신 전 계수: {w:.4f}')
print(f'갱신 후 계수: {w_new:.4f}')
print(f'갱신 전 절편: {b:.4f}')
print(f'갱신 후 절편: {b_new:.4f}')

### B-4. 갱신 후 예측·손실 비교

In [ ]:
# 수정된 계수와 절편으로 같은 샘플을 다시 예측
z_new = w_new * x + b_new
p_new = 1 / (1 + np.exp(-z_new))
loss_new = -np.log(p_new)

results_b = pd.DataFrame({
    '구분': ['갱신 전', '갱신 후'],
    '계수': [w, w_new],
    '절편': [b, b_new],
    '예측 확률': [p, p_new],
    '손실': [loss, loss_new]
})
print(results_b.round(4).to_string(index=False))

### B-5. 응용

B-1 셀의 `learning_rate`를 `0.1`에서 `0.5`로 **직접 바꾸고 B-1~B-4를 다시 실행**합니다. 기본값으로 되돌린 후 다음 실습을 진행합니다.

## 교재 실습 연결 — 7-2, 7-3

교안의 `SGDClassifier` 기본 학습과 `partial_fit()`을 실행합니다. 교재의 출력과 다르면 라이브러리 버전을 확인합니다.

### 7-2. SGDClassifier 학습

In [ ]:
from sklearn.linear_model import SGDClassifier

sc = SGDClassifier(
    loss='log_loss',
    max_iter=10,
    random_state=42
)

sc.fit(train_scaled, train_target)

print(sc.score(train_scaled, train_target))
print(sc.score(test_scaled, test_target))

### 7-3. partial_fit() 한 에포크 추가 학습

In [ ]:
sc.partial_fit(train_scaled, train_target)

print(sc.score(train_scaled, train_target))
print(sc.score(test_scaled, test_target))

## 실습 C — partial_fit() 전후 모델 파라미터 비교 (교안 7-4)

### C-1. 비교 전용 모델 학습·상태 저장

In [ ]:
sc_compare = SGDClassifier(
    loss='log_loss',
    max_iter=10,
    random_state=42
)
sc_compare.fit(train_scaled, train_target)

# 추가 학습 전의 계수·절편을 복사해 둠
coef_before = sc_compare.coef_.copy()
intercept_before = sc_compare.intercept_.copy()

train_score_before = sc_compare.score(train_scaled, train_target)
test_score_before = sc_compare.score(test_scaled, test_target)

print('계수 배열 크기:', coef_before.shape)
print('절편 배열 크기:', intercept_before.shape)
print('추가 학습 전 훈련 정확도:', round(train_score_before, 4))
print('추가 학습 전 테스트 정확도:', round(test_score_before, 4))

### C-2. 한 에포크 추가 학습

In [ ]:
sc_compare.partial_fit(train_scaled, train_target)

# 추가 학습 후의 계수·절편
coef_after = sc_compare.coef_.copy()
intercept_after = sc_compare.intercept_.copy()

train_score_after = sc_compare.score(train_scaled, train_target)
test_score_after = sc_compare.score(test_scaled, test_target)

print('추가 학습 후 훈련 정확도:', round(train_score_after, 4))
print('추가 학습 후 테스트 정확도:', round(test_score_after, 4))

### C-3. 계수와 절편 변화 확인

In [ ]:
coef_change = np.abs(coef_after - coef_before)
intercept_change = np.abs(intercept_after - intercept_before)

print('계수가 변경되었나요?', not np.allclose(coef_before, coef_after))
print('절편이 변경되었나요?', not np.allclose(intercept_before, intercept_after))
print('계수의 평균 절댓값 변화:', round(coef_change.mean(), 4))
print('절편의 평균 절댓값 변화:', round(intercept_change.mean(), 4))

### C-3. 훈련·테스트 정확도 변화

In [ ]:
results_c = pd.DataFrame({
    '평가 항목': ['훈련 정확도', '테스트 정확도'],
    '추가 학습 전': [train_score_before, test_score_before],
    '추가 학습 후': [train_score_after, test_score_after]
})
print(results_c.round(4).to_string(index=False))

### C-4. 응용 — 두 번째 추가 학습

**C-1은 다시 실행하지 않습니다.** 아래 셀에서 현재 파라미터와 정확도를 비교 기준으로 저장한 다음, **C-2와 C-3의 코드 셀들을 다시 실행**합니다.

In [ ]:
# 현재 모델의 상태를 다음 추가 학습의 비교 기준으로 저장
coef_before = sc_compare.coef_.copy()
intercept_before = sc_compare.intercept_.copy()

train_score_before = sc_compare.score(train_scaled, train_target)
test_score_before = sc_compare.score(test_scaled, test_target)

## 8. 에포크와 일반화 — 훈련·검증·테스트 구분

### 8-2. 훈련 데이터 내부에서 검증 세트 분리

In [ ]:
sub_input, val_input, sub_target, val_target = train_test_split(
    train_input,
    train_target,
    test_size=0.2,
    random_state=42,
    stratify=train_target
)

print('학습용 데이터:', sub_input.shape)
print('검증 데이터:', val_input.shape)

### 8-2. 학습용 데이터로만 표준화 기준 계산

In [ ]:
ss_val = StandardScaler()
ss_val.fit(sub_input)

sub_scaled = ss_val.transform(sub_input)
val_scaled = ss_val.transform(val_input)

### 8-3. 300에포크 실험 준비

In [ ]:
import numpy as np

sc_epoch = SGDClassifier(
    loss='log_loss',
    random_state=42
)

train_score = []
val_score = []

classes = np.unique(sub_target)

### 8-3. 에포크마다 훈련·검증 정확도 기록

테스트 세트는 학습 횟수 선택에 사용하지 않습니다.

In [ ]:
for _ in range(300):
    sc_epoch.partial_fit(
        sub_scaled,
        sub_target,
        classes=classes
    )

    train_score.append(
        sc_epoch.score(sub_scaled, sub_target)
    )

    val_score.append(
        sc_epoch.score(val_scaled, val_target)
    )

### 8-4. 훈련·검증 정확도 그래프

In [ ]:
import matplotlib.pyplot as plt

epochs = np.arange(1, 301)

plt.plot(epochs, train_score, label='Train')
plt.plot(epochs, val_score, label='Validation')

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

### 8-5. 검증 최고점 확인

In [ ]:
best_epoch = int(np.argmax(val_score) + 1)
best_val_score = float(np.max(val_score))

print('가장 높은 검증 정확도의 에포크:', best_epoch)
print('가장 높은 검증 정확도:', round(best_val_score, 4))

### 8-5. 11·30·60·100·200·300에포크 결과 비교

In [ ]:
for epoch in [11, 30, 60, 100, 200, 300]:
    print(f'{epoch:3d}에포크  훈련 {train_score[epoch - 1]:.4f}  검증 {val_score[epoch - 1]:.4f}')

### 8-5. 학습 횟수 선택

교안에서는 최고점 한 번만으로 판단하지 않고 **실습용으로 100에포크**를 선택했습니다.

In [ ]:
selected_epoch = 100

### 8-6. 전체 훈련 데이터로 최종 재학습

In [ ]:
sc_final = SGDClassifier(
    loss='log_loss',
    max_iter=selected_epoch,
    tol=None,
    random_state=42
)

sc_final.fit(train_scaled, train_target)

print('최종 모델의 훈련 정확도:',
      round(sc_final.score(train_scaled, train_target), 4))

### 8-7. 테스트 세트 최종 평가

이 모델의 에포크는 검증 결과로 선택했습니다. 교재 재현을 위한 다른 테스트 출력과 혼동하지 않습니다.

In [ ]:
final_test_score = sc_final.score(
    test_scaled,
    test_target
)

print('최종 테스트 정확도:', round(final_test_score, 4))

## 9. 손실 함수 변경 — 교재 재현과 검증 비교

### 9-1. 교재의 hinge 손실 실습 재현

**교육용 비교:** 아래 테스트 결과로 다시 모델을 선택하지 않습니다.

In [ ]:
sc = SGDClassifier(
    loss='hinge',
    max_iter=100,
    tol=None,
    random_state=42
)

sc.fit(train_scaled, train_target)

print(sc.score(train_scaled, train_target))
print(sc.score(test_scaled, test_target))

### 9-2. 검증 세트로 log_loss와 hinge 비교

In [ ]:
sc_log = SGDClassifier(
    loss='log_loss',
    max_iter=100,
    tol=None,
    random_state=42
)

sc_hinge = SGDClassifier(
    loss='hinge',
    max_iter=100,
    tol=None,
    random_state=42
)

sc_log.fit(sub_scaled, sub_target)
sc_hinge.fit(sub_scaled, sub_target)

results_loss = pd.DataFrame({
    '손실 함수': ['log_loss', 'hinge'],
    '훈련 정확도': [
        sc_log.score(sub_scaled, sub_target),
        sc_hinge.score(sub_scaled, sub_target)
    ],
    '검증 정확도': [
        sc_log.score(val_scaled, val_target),
        sc_hinge.score(val_scaled, val_target)
    ]
})

print(results_loss.round(4).to_string(index=False))